# ModelListGP: independent multi-output surrogate

`ModelListGP` は複数の独立した GP を1つのmulti-output modelとしてまとめます。出力ごとに観測位置やmodel contractが異なる場合に有効です。

## このNotebookで確認すること

- 異なる入力位置で観測された2出力の構築
- 子`SingleTaskGP`を束ねる`ModelListGP`
- `SumMarginalLogLikelihood`による一括学習
- multi-output posteriorのshapeと可視化
- MultiTask GPとの意味上の違い
- 2目的の獲得関数、候補最適化、目的関数の評価

関連理論:

- [Multi-task / multi-output](../../docs/theory/07_multitask_multioutput.md)
- [Model selection](../../docs/theory/13_model_selection.md)

このNotebookでは独立した2つのGPから、多目的ベイズ最適化の候補を1点提案します。

## 1. Import と再現性設定

In [ ]:
import matplotlib.pyplot as plt
import torch
from botorch.acquisition.multi_objective.logei import qLogExpectedHypervolumeImprovement
from botorch.optim import optimize_acqf
from botorch.utils.multi_objective.box_decompositions.non_dominated import NondominatedPartitioning

from robotorchan.models import ModelListGP, SingleTaskGP

torch.manual_seed(0)
dtype = torch.double

### 独立した2つの目的を最適化する

ここでは $f_1(x)$ と $f_2(x)$ を**両方最大化**します。一般に、片方の目的を改善するともう片方が悪化することがあるため、最適解が1点に定まるとは限りません。どちらの目的でも他の点に劣らない観測を**非支配解（パレート解）**と呼びます。

Notebook 04の `KroneckerMultiTaskGP` は同じ入力で全taskを観測するblock-designでした。`ModelListGP` は**出力ごとに異なる入力で学習できる**のが特徴で、目的間の相関は原則として学習しません。

## 2. 異なる学習入力を持つ2つの出力

2つの目的関数を、それぞれ異なる `X` の位置で観測したデータを作ります。

In [ ]:
def objective_1(x: torch.Tensor) -> torch.Tensor:
    return torch.sin(2 * torch.pi * x)

def objective_2(x: torch.Tensor) -> torch.Tensor:
    return 0.6 * torch.cos(2 * torch.pi * x) + 0.2 * x

train_X1 = torch.linspace(0.05, 0.95, 10, dtype=dtype).unsqueeze(-1)
train_X2 = torch.linspace(0.10, 0.90, 7, dtype=dtype).unsqueeze(-1)

train_Y1 = objective_1(train_X1) + 0.03 * torch.randn_like(train_X1)
train_Y2 = objective_2(train_X2) + 0.03 * torch.randn_like(train_X2)

### 出力ごとに観測場所が違う例

`train_X1` は10点、`train_X2` は7点です。`ModelListGP` では両目的を同じ入力で測定していなくても、それぞれのGPを学習できます。

`train_Y1` と `train_Y2` には再現可能な乱数ノイズを加えています。一方、下の候補評価は説明を簡単にするためノイズなしの目的関数を使います。実際には評価コストやノイズ水準が目的ごとに異なる場合もあります。

## 3. 子モデルと `ModelListGP` の構築

各出力に `SingleTaskGP` を作り、それらを `ModelListGP` にまとめます。

In [ ]:
model_1 = SingleTaskGP(train_X1, train_Y1)
model_2 = SingleTaskGP(train_X2, train_Y2)
model = ModelListGP(model_1, model_2)

print("number of child models:", len(model.models))
print("supports_mll:", model.supports_mll)
print("raw_train_Xs shapes:", [x.shape for x in model.raw_train_Xs])
print("raw_train_Ys shapes:", [y.shape for y in model.raw_train_Ys])

`ModelListGP` は raw data を子モデルごとに保持します。コンテナ全体に対する単一の `raw_train_X` を新たに作ることはせず、`raw_train_Xs` / `raw_train_Ys` として公開します。

## 4. すべての子モデルをまとめて学習

`make_mll()` は `SumMarginalLogLikelihood` を返すため、`model.fit()` で子モデルをまとめて学習できます。内部ではBoTorchの `fit_gpytorch_mll()` を使用します。

In [ ]:
mll = model.make_mll()
print(type(mll).__name__)

model.fit()

### `SumMarginalLogLikelihood` の意味

各子モデルの周辺対数尤度を合計して最適化することで、独立したGPをまとめて学習します。`model.fit()` はこの処理を実行します。出力間の共分散を学習するわけではないため、**目的間の情報共有を期待する場合はMulti-task GPとの使い分け**が必要です。

## 5. Multi-output posterior

In [ ]:
test_X = torch.linspace(0.0, 1.0, 200, dtype=dtype).unsqueeze(-1)

model.eval()
with torch.no_grad():
    posterior = model.posterior(test_X)
    mean = posterior.mean
    variance = posterior.variance

print("posterior mean shape:", mean.shape)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))

ax.scatter(train_X1.squeeze(-1), train_Y1.squeeze(-1), label="output 1 observations")
ax.scatter(train_X2.squeeze(-1), train_Y2.squeeze(-1), label="output 2 observations")
ax.plot(test_X.squeeze(-1), mean[..., 0], label="output 1 posterior")
ax.plot(test_X.squeeze(-1), mean[..., 1], label="output 2 posterior")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_title("ModelListGP with independent outputs")
ax.legend()
plt.show()

### 追加実装例：目的ごとの予測不確実性

`posterior.variance` から各目的の標準偏差を計算し、平均と約95%の**潜在関数に関するモデル上の区間**を別々に描きます。これは将来の測定値の予測区間や、真の関数を必ず含む保証とは異なります。

In [ ]:
std = variance.clamp_min(0).sqrt()
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharex=True)
for i, (ax, train_x, train_y) in enumerate(
    zip(axes, (train_X1, train_X2), (train_Y1, train_Y2))
):
    ax.scatter(train_x.squeeze(-1), train_y.squeeze(-1), label="observations")
    ax.plot(test_X.squeeze(-1), mean[..., i], label="posterior mean")
    ax.fill_between(
        test_X.squeeze(-1),
        mean[..., i] - 1.96 * std[..., i],
        mean[..., i] + 1.96 * std[..., i],
        alpha=0.2,
        label="approx. 95% latent interval",
    )
    ax.set_title(f"objective {i + 1}")
    ax.set_xlabel("x")
    ax.legend()
axes[0].set_ylabel("objective value")
plt.show()

## 6. Multi-task GP ではなく `ModelListGP` を使う理由

`ModelListGP` の子モデルは原則として独立です。出力ごとに観測位置が異なる、出力間相関をモデル化する必要がない、出力ごとに異なる GP や transform を使いたい、multi-objective BO で目的ごとに別 surrogate を持ちたい、といった場合に適しています。

タスク間 covariance を学習して情報共有すること自体が目的なら `MultiTaskGP` や `KroneckerMultiTaskGP` を検討します。

### qLogEHVIでパレート探索する理由

`qLogExpectedHypervolumeImprovement`（qLogEHVI）は、参照点から測った**非支配領域のハイパーボリュームの期待改善**を利用して、複数目的のトレードオフを探索します。

ここでは異なる入力位置で学習した2つのGPの予測平均を、共通の `reference_X` 上で評価して非支配分割を作っています。これは**モデル予測に基づく近似的な参照集合**であり、実測した同時観測のパレート集合ではありません。厳密に観測済みのハイパーボリューム改善を基準にする場合は、両目的が実測された共通入力の `Y` から分割を作る必要があります。

## 7. Multi-objective candidate generation

各目的関数は最大化します。参照点と非支配分割は両目的を同じ入力点で評価した値から作ります。これは候補生成のための小規模な例であり、異なる観測位置に対する既存の子モデル学習は維持します。

In [ ]:
bounds = torch.tensor([[0.0], [1.0]], dtype=dtype)
reference_X = torch.linspace(0.0, 1.0, 21, dtype=dtype).unsqueeze(-1)
with torch.no_grad():
    reference_Y = model.posterior(reference_X).mean

ref_point = (reference_Y.min(dim=0).values - 0.1).tolist()
partitioning = NondominatedPartitioning(
    ref_point=torch.tensor(ref_point, dtype=dtype),
    Y=reference_Y,
)
acquisition = qLogExpectedHypervolumeImprovement(
    model=model,
    ref_point=ref_point,
    partitioning=partitioning,
)
candidate, acquisition_value = optimize_acqf(
    acq_function=acquisition,
    bounds=bounds,
    q=1,
    num_restarts=5,
    raw_samples=64,
)
assert candidate.shape == (1, 1)
assert torch.all((candidate >= bounds[0]) & (candidate <= bounds[1]))
print("candidate:", candidate)
print("acquisition value:", acquisition_value)


## 8. Candidate evaluation

提案された入力を2つの目的関数で評価します。実際の最適化では、この結果を各子モデルの観測データに追加して再学習します。

In [ ]:
candidate_Y = torch.cat([objective_1(candidate), objective_2(candidate)], dim=-1)
print("candidate objectives:", candidate_Y)
assert candidate_Y.shape == (1, 2)


In [ ]:
from botorch.utils.multi_objective.pareto import is_non_dominated

predicted_pareto_mask = is_non_dominated(reference_Y)
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(reference_Y[:, 0], reference_Y[:, 1], alpha=0.35, label="posterior mean grid")
ax.scatter(
    reference_Y[predicted_pareto_mask, 0],
    reference_Y[predicted_pareto_mask, 1],
    marker="D",
    label="predicted non-dominated grid",
)
ax.scatter(
    candidate_Y[:, 0], candidate_Y[:, 1],
    marker="*", s=160, label="newly observed candidate",
)
ax.set_xlabel("objective 1 (maximize)")
ax.set_ylabel("objective 2 (maximize)")
ax.legend()
plt.show()

## 10. 追加実装例：観測したパレート解の可視化

初期の2目的は異なる入力で観測したため、**初期データ全体をそのまま2次元の観測済みパレート集合として扱うことはできません**。ここでは両目的を実際に評価した新しい候補を可視化し、比較用に共通グリッド上の**予測平均による近似パレート集合**を表示します。

予測平均から得た非支配点は実測パレート解ではありません。観測済みのパレート集合を充実させるには、両目的を同時に測る候補を繰り返し追加してください。

In [ ]:
updated_X1 = torch.cat([train_X1, candidate], dim=0)
updated_Y1 = torch.cat([train_Y1, candidate_Y[:, :1]], dim=0)
updated_X2 = torch.cat([train_X2, candidate], dim=0)
updated_Y2 = torch.cat([train_Y2, candidate_Y[:, 1:2]], dim=0)

updated_model = ModelListGP(
    SingleTaskGP(updated_X1, updated_Y1),
    SingleTaskGP(updated_X2, updated_Y2),
)
updated_model.fit()
updated_model.eval()

assert updated_X1.shape[0] == train_X1.shape[0] + 1
assert updated_X2.shape[0] == train_X2.shape[0] + 1
with torch.no_grad():
    updated_candidate_mean = updated_model.posterior(candidate).mean
print("updated posterior mean at candidate:", updated_candidate_mean)

## 9. 追加実装例：観測した候補を両方の子モデルへ追加する

上の候補は両目的を評価したので、`candidate` をそれぞれの学習入力へ、`candidate_Y` の対応列を各出力へ追加できます。これにより、**目的ごとの観測数が異なるまま**データを更新できます。次のコードでは、更新した `ModelListGP` を再学習し、次回の予測に反映させます。